# 6.3 Apply the dedup rule and re-measure MAE

Same split and CatBoost settings as 5.3. Duplicates are removed after the split, so train / val / test keep the same groups. Needs `dup_rate_6_2.csv` from 6.2.

In [1]:
import numpy as np
import pandas as pd

In [27]:
df_train = pd.read_csv('../data/processed/task6/train_cleaned6.csv')
df_val =pd.read_csv('../data/processed/task6/val_cleaned6.csv')


print("Dataset shape:", df_train.shape)
print("Dataset shape:", df_val.shape)



Dataset shape: (5048, 70)
Dataset shape: (1249, 70)


In [28]:

print("Columns:")
print(df_train.columns.tolist())

print("\nMissing values in important columns:")
important_cols = [
    col for col in [
        "ad_id",
        "price",
        "model_key",
        "title_group",
        "dup_candidate_id"
    ]
    if col in df_train.columns
]

display(df_train[important_cols].isna().sum().to_frame("missing_values"))


Columns:
['title', 'price', 'location', 'ad_url', 'ad_id', 'run_id', 'brand', 'storage_gb', 'ram_gb', 'storage_src', 'condition', 'condition_src', 'ram_src', 'has_5g', 'has_warranty', 'has_box', 'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint', 'is_tablet_hint', 'is_parts_hint', 'accessory_first', 'bundle_hint', 'is_feature_phone_hint', 'price_placeholder', 'loc_area', 'loc_city', 'price_missing', 'price_suspicious', 'price_very_high', 'title_normalized', 'title_punct_normalized', 'title_group', 'is_phone_listing', 'lg_model', 'model_key', 'match_rule', 'confidence', 'family', 'storage_known', 'ram_known', 'condition_known', 'storage_imp_src', 'storage_imp_conf', 'storage_imp_post', 'ram_imp_src', 'ram_imp_conf', 'ram_imp_post', 'imp_price_used', 'ref_matched', 'storage_gb_orig', 'storage_gb_was_imputed', 'ram_gb_orig', 'ram_gb_was_imputed', 'ref_n', 'ref_med_log', 'ref_mad_log', 'ref_level', 'ref_median_price', 'price_ratio', 'mad_z', 'price_review_reason', 'price_flag', 'pric

,missing_values
ad_id,0
price,0
model_key,0
title_group,0
dup_candidate_id,4534


In [29]:

required_cols = ["dup_candidate_id", "loc_area"]

missing_cols = [
    col for col in required_cols
    if col not in df_train.columns
]

if missing_cols:
    print("Missing columns:", missing_cols)
else:
    valid = (
        df_train["dup_candidate_id"].notna()
        & df_train["loc_area"].notna()
    )

    remaining_duplicates = df_train.loc[valid].duplicated(
        subset=["dup_candidate_id", "loc_area"],
        keep=False
    ).sum()

    print("Ads involved in remaining duplicate pairs:", remaining_duplicates)


Ads involved in remaining duplicate pairs: 0


In [ ]:

valid = (df_train["dup_candidate_id"].notna()& df_train["loc_area"].notna())

drop_mask = pd.Series(False, index=df_train.index)

drop_mask.loc[valid] = df_train.loc[valid].duplicated(subset=["dup_candidate_id", "loc_area"],keep="first")

print("Total ads:", len(df_train))
print("Ads involved in duplicate groups:", int(
    df_train.loc[valid].duplicated(
        subset=["dup_candidate_id", "loc_area"],
        keep=False
    ).sum()
))
print("Ads to remove:", int(drop_mask.sum()))
print("Ads remaining:", int((~drop_mask).sum()))


Total ads: 5048
Ads involved in duplicate groups: 0
Ads to remove: 0
Ads remaining: 5048


-------------------------------------------------------------------------

In [31]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import mean_absolute_error
from catboost import CatBoostRegressor


features = [
    'model_key', 'brand', 'family', 'storage_gb', 'ram_gb',
    'condition', 'has_5g', 'has_warranty', 'has_box',
    'mentions_exchange', 'is_wanted_ad', 'is_accessory_hint',
    'is_tablet_hint', 'is_parts_hint', 'accessory_first',
    'bundle_hint', 'is_feature_phone_hint', 'loc_area',
    'loc_city'
]

cat_features = [
    'model_key', 'brand', 'family',
    'condition', 'loc_area', 'loc_city'
]

catboost_params = {
    'iterations': 1500,
    'learning_rate': 0.03,
    'depth': 6,
    'eval_metric': 'MAE',
    'random_seed': 42,
    'verbose': 0,
    'allow_writing_files': False
}

missing_features = [c for c in features if c not in df_train.columns]

if missing_features:
    raise ValueError(f"Missing features: {missing_features}")

print("Features:", len(features))
print("Categorical features:", len(cat_features))


Features: 19
Categorical features: 6


In [32]:

def prepare(data):
    X = data[features].copy()
    X[cat_features] = (X[cat_features].fillna("Unknown").astype(str))
    return X

def train_model(train, val):

    # Validation rows with valid prices
    early_stop = val[(val['listing_type'] == 'phone') &val['price'].notna()].copy()
    if early_stop.empty:
        raise ValueError(
            "No phone listings with valid prices in validation data."
        )

    model = CatBoostRegressor(loss_function='Quantile:alpha=0.5',**catboost_params)

    model.fit(prepare(train),train['price'],cat_features=cat_features,
        eval_set=(
            prepare(early_stop),
            early_stop['price']
        ),
        early_stopping_rounds=100,
        verbose=False
    )

    return model


In [33]:
def calculate_grouped_mae(y_true, y_pred, groups):

    evaluation = pd.DataFrame({
        'actual': np.asarray(y_true),
        'predicted': np.asarray(y_pred),
        'group': np.asarray(groups)
    }).dropna(subset=['group'])

    if evaluation.empty:
        return np.nan

    evaluation['absolute_error'] = (evaluation['actual'] - evaluation['predicted']
).abs()

    return (
        evaluation.groupby('group')['absolute_error']
        .mean()
        .mean()
    )



In [ ]:
results = []

for seed in [42, 7, 2026]:
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.50,
        random_state=seed
    )

    early_idx, test_idx = next(
        splitter.split(df_val, groups=df_val['title_group'])
    )

    val_early = df_val.iloc[early_idx].copy()
    test = df_val.iloc[test_idx].copy()

    # Evaluate on phone listings with valid prices only
    test_phone = test[(test['listing_type'] == 'phone') &test['price'].notna()].copy()
    model = train_model(df_train, val_early)
    predictions = model.predict(prepare(test_phone))

    mae = mean_absolute_error(test_phone['price'], predictions)

    grouped_mae = calculate_grouped_mae(test_phone['price'],predictions,test_phone['title_group'])

    results.append({
        'seed': seed,
        'train_size': len(df_train),
        'validation_size': len(val_early),
        'test_size': len(test),
        'test_phone_ads': len(test_phone),
        'test_title_groups': test_phone['title_group'].nunique(),
        'MAE_clean': mae,
        'Grouped_MAE_clean': grouped_mae
    })

results_df = pd.DataFrame(results)

display(results_df.round(2))

print("\nAverage results across seeds:")
display(
    results_df.drop(columns='seed')
    .mean()
    .to_frame('Average')
    .round(2)
)

,seed,train_size,validation_size,test_size,test_phone_ads,test_title_groups,MAE_clean,Grouped_MAE_clean
0,42,5048,651,598,544,230,5951.63,5389.11
1,7,5048,554,695,650,238,6948.81,7914.51
2,2026,5048,685,564,521,238,5338.14,6007.44



Average results across seeds:


,Average
train_size,5048.00
validation_size,630.00
test_size,619.00
test_phone_ads,571.67
test_title_groups,235.33
MAE_clean,6079.53
Grouped_MAE_clean,6437.02
